# Reproduction notebook for arXiv:2603.06431 (Lp + W1p only)

This notebook reproduces the paper's **norm-bound experiments** for:

- $L^p$ (Section 4.5.2 and part of 4.5.3)
- $W^{1,p}$ (Section 4.5.1 and part of 4.5.3)

and **intentionally excludes $W^{2,p}$**.

It mirrors the paper's experiment structure:

1. 1D random (untrained) deep vs wide networks,
2. 1D trained deep vs wide networks,
3. 2D trained deep vs wide networks,
4. figure-style convergence curves with mean + 95% confidence intervals.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import math
import random
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from intervalnets import IntervalTensor, enable_interval_eval

enable_interval_eval()


## Configuration (paper-like defaults)

The defaults below follow the paper setup style (100 runs and long training). If you want a quick run,
set `QUICK_MODE = True`.


In [ ]:
# Reproducibility
BASE_SEED = 1234

# Paper-like settings
PAPER_RUNS = 100
PAPER_TRAIN_EPOCHS_1D = 2000
PAPER_TRAIN_EPOCHS_2D_LP = 2000
PAPER_TRAIN_EPOCHS_2D_W1P = 10000

# Practical override for local experimentation
QUICK_MODE = False
QUICK_RUNS = 10
QUICK_EPOCHS_1D = 200
QUICK_EPOCHS_2D_LP = 300
QUICK_EPOCHS_2D_W1P = 500

N_RUNS = QUICK_RUNS if QUICK_MODE else PAPER_RUNS
EPOCHS_1D = QUICK_EPOCHS_1D if QUICK_MODE else PAPER_TRAIN_EPOCHS_1D
EPOCHS_2D_LP = QUICK_EPOCHS_2D_LP if QUICK_MODE else PAPER_TRAIN_EPOCHS_2D_LP
EPOCHS_2D_W1P = QUICK_EPOCHS_2D_W1P if QUICK_MODE else PAPER_TRAIN_EPOCHS_2D_W1P

P_VAL = 2.0
ITERATIONS = list(range(0, 11))

print(f"QUICK_MODE={QUICK_MODE}")
print(f"N_RUNS={N_RUNS}, EPOCHS_1D={EPOCHS_1D}, EPOCHS_2D_LP={EPOCHS_2D_LP}, EPOCHS_2D_W1P={EPOCHS_2D_W1P}")

## Helpers

In [ ]:
@dataclass
class Arch:
    name: str
    input_dim: int
    hidden_layers: int
    width: int
    activation: str


def make_network(arch: Arch) -> nn.Sequential:
    act = nn.Tanh if arch.activation == "tanh" else nn.ReLU
    layers = []
    in_dim = arch.input_dim
    for _ in range(arch.hidden_layers):
        layers.append(nn.Linear(in_dim, arch.width))
        layers.append(act())
        in_dim = arch.width
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def ci95(x: np.ndarray):
    # 95% confidence interval of the mean (normal approximation)
    m = x.mean(axis=0)
    s = x.std(axis=0, ddof=1)
    n = x.shape[0]
    h = 1.96 * s / np.sqrt(max(n, 1))
    return m, m - h, m + h


def gaussian_peak_1d(x: torch.Tensor) -> torch.Tensor:
    # Localized 1D Gaussian-like peak used as training target in the paper's 1D experiments.
    return torch.exp(-40.0 * x.pow(2))


def smooth_disk_2d(xy: torch.Tensor, radius: float = 0.6) -> torch.Tensor:
    # Compactly supported smooth bump (disk-like profile) for 2D experiments.
    r2 = xy[:, 0].pow(2) + xy[:, 1].pow(2)
    z = torch.zeros_like(r2)
    inside = r2 < radius * radius
    t = 1.0 - r2[inside] / (radius * radius)
    z[inside] = torch.exp(-1.0 / torch.clamp(t, min=1e-8))
    return z


def train_to_target(model: nn.Module, dim: int, target_fn, epochs: int, lr: float = 1e-3, weight_decay: float = 0.0, batch_size: int = 2048):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        x = torch.rand(batch_size, dim) * 2.0 - 1.0
        y = target_fn(x if dim > 1 else x[:, :1])
        pred = model(x).squeeze(-1)
        loss = torch.mean((pred - y) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()


def mc_lp(model: nn.Module, dim: int, p: float, n: int = 50_000) -> float:
    with torch.no_grad():
        x = torch.rand(n, dim) * 2.0 - 1.0
        y = model(x).squeeze(-1).abs().pow(p)
        integral = (2.0 ** dim) * y.mean().item()
        return integral ** (1.0 / p)


def mc_w1p(model: nn.Module, dim: int, p: float, n: int = 50_000) -> float:
    x = torch.rand(n, dim, requires_grad=True) * 2.0 - 1.0
    y = model(x).squeeze(-1)
    grad = torch.autograd.grad(y.sum(), x, create_graph=False)[0]
    integrand = y.abs().pow(p) + torch.linalg.vector_norm(grad, ord=2, dim=-1).pow(p)
    integral = (2.0 ** dim) * integrand.mean().item()
    return integral ** (1.0 / p)


def bound_gap_curve(model: nn.Module, domain: IntervalTensor, p: float, mode: str, iterations: list[int], ref_value: float) -> np.ndarray:
    vals = []
    for it in iterations:
        if mode == "lp":
            b = model.lpnorm(domain, p=p, iterations=it)
        elif mode == "w1p":
            b = model.sobolev_norm(domain, p=p, iterations=it)
        else:
            raise ValueError(mode)
        vals.append((float(b.upper - b.lower)) / max(ref_value, 1e-12))
    return np.array(vals, dtype=float)


## 1D setups (deep vs wide)

In [ ]:
deep_tanh_1d = Arch(name="deep", input_dim=1, hidden_layers=3, width=32, activation="tanh")
wide_tanh_1d = Arch(name="wide", input_dim=1, hidden_layers=1, width=200, activation="tanh")

deep_relu_1d = Arch(name="deep", input_dim=1, hidden_layers=3, width=32, activation="relu")
wide_relu_1d = Arch(name="wide", input_dim=1, hidden_layers=1, width=200, activation="relu")

domain_1d = IntervalTensor.from_bounds([-1.0], [1.0])

def run_family(arch: Arch, mode: str, trained: bool):
    curves = []
    for run in range(N_RUNS):
        set_seed(BASE_SEED + run)
        model = make_network(arch)

        if trained:
            if mode == "w1p":
                train_to_target(model, dim=1, target_fn=gaussian_peak_1d, epochs=EPOCHS_1D, lr=1e-3)
                ref = mc_w1p(model, dim=1, p=P_VAL, n=50_000)
            else:
                train_to_target(model, dim=1, target_fn=gaussian_peak_1d, epochs=EPOCHS_1D, lr=1e-3)
                ref = mc_lp(model, dim=1, p=P_VAL, n=50_000)
        else:
            ref = mc_w1p(model, dim=1, p=P_VAL, n=50_000) if mode == "w1p" else mc_lp(model, dim=1, p=P_VAL, n=50_000)

        curve = bound_gap_curve(model, domain_1d, p=P_VAL, mode=mode, iterations=ITERATIONS, ref_value=ref)
        curves.append(curve)

    return np.stack(curves, axis=0)


## Figure A — 1D Sobolev ($W^{1,p}$) mean normalized global bound gap (random vs trained)

In [ ]:
w1p_deep_untrained = run_family(deep_tanh_1d, mode="w1p", trained=False)
w1p_wide_untrained = run_family(wide_tanh_1d, mode="w1p", trained=False)

w1p_deep_trained = run_family(deep_tanh_1d, mode="w1p", trained=True)
w1p_wide_trained = run_family(wide_tanh_1d, mode="w1p", trained=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], w1p_deep_untrained, w1p_wide_untrained, "Untrained tanh networks"),
    (axes[1], w1p_deep_trained, w1p_wide_trained, "Trained tanh networks (Gaussian peak)"),
]:
    for label, arr, color in [("deep (3x32)", deep_arr, "tab:blue"), ("wide (1x200)", wide_arr, "tab:orange")]:
        m, lo, hi = ci95(arr)
        ax.plot(ITERATIONS, m, color=color, label=label)
        ax.fill_between(ITERATIONS, lo, hi, color=color, alpha=0.2)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("1D W1p reproduction (paper-style)")
plt.tight_layout()
plt.show()

## Figure B — 1D Lebesgue ($L^p$) mean normalized global bound gap (random vs trained)

In [ ]:
lp_deep_untrained = run_family(deep_relu_1d, mode="lp", trained=False)
lp_wide_untrained = run_family(wide_relu_1d, mode="lp", trained=False)

lp_deep_trained = run_family(deep_relu_1d, mode="lp", trained=True)
lp_wide_trained = run_family(wide_relu_1d, mode="lp", trained=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], lp_deep_untrained, lp_wide_untrained, "Untrained ReLU networks"),
    (axes[1], lp_deep_trained, lp_wide_trained, "Trained ReLU networks (Gaussian peak)"),
]:
    for label, arr, color in [("deep (3x32)", deep_arr, "tab:green"), ("wide (1x200)", wide_arr, "tab:red")]:
        m, lo, hi = ci95(arr)
        ax.plot(ITERATIONS, m, color=color, label=label)
        ax.fill_between(ITERATIONS, lo, hi, color=color, alpha=0.2)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("1D Lp reproduction (paper-style)")
plt.tight_layout()
plt.show()

## Figure C — 2D trained networks (deep vs wide), $L^p$ and $W^{1,p}$ convergence

In [ ]:
deep_relu_2d = Arch(name="deep", input_dim=2, hidden_layers=3, width=32, activation="relu")
wide_relu_2d = Arch(name="wide", input_dim=2, hidden_layers=1, width=200, activation="relu")

deep_tanh_2d = Arch(name="deep", input_dim=2, hidden_layers=3, width=32, activation="tanh")
wide_tanh_2d = Arch(name="wide", input_dim=2, hidden_layers=1, width=200, activation="tanh")

domain_2d = IntervalTensor.from_bounds([-1.0, -1.0], [1.0, 1.0])

# Lp (trained ReLU)
set_seed(BASE_SEED + 10_000)
lp_deep_2d = make_network(deep_relu_2d)
train_to_target(lp_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP, lr=1e-3, weight_decay=0.0)
lp_ref_deep = mc_lp(lp_deep_2d, dim=2, p=P_VAL, n=50_000)
lp_curve_deep = bound_gap_curve(lp_deep_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_deep)

set_seed(BASE_SEED + 10_001)
lp_wide_2d = make_network(wide_relu_2d)
train_to_target(lp_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP, lr=1e-3, weight_decay=0.0)
lp_ref_wide = mc_lp(lp_wide_2d, dim=2, p=P_VAL, n=50_000)
lp_curve_wide = bound_gap_curve(lp_wide_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_wide)

# W1p (trained tanh)
set_seed(BASE_SEED + 10_100)
w1p_deep_2d = make_network(deep_tanh_2d)
train_to_target(w1p_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P, lr=1e-3, weight_decay=1e-6)
w1p_ref_deep = mc_w1p(w1p_deep_2d, dim=2, p=P_VAL, n=50_000)
w1p_curve_deep = bound_gap_curve(w1p_deep_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1p_ref_deep)

set_seed(BASE_SEED + 10_101)
w1p_wide_2d = make_network(wide_tanh_2d)
train_to_target(w1p_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P, lr=1e-3, weight_decay=0.0)
w1p_ref_wide = mc_w1p(w1p_wide_2d, dim=2, p=P_VAL, n=50_000)
w1p_curve_wide = bound_gap_curve(w1p_wide_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1p_ref_wide)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(ITERATIONS, lp_curve_deep, marker="o", label="deep (3x32)")
axes[0].plot(ITERATIONS, lp_curve_wide, marker="o", label="wide (1x200)")
axes[0].set_title("2D trained Lp (ReLU)")
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel("refinement iterations")
axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()

axes[1].plot(ITERATIONS, w1p_curve_deep, marker="o", label="deep (3x32)")
axes[1].plot(ITERATIONS, w1p_curve_wide, marker="o", label="wide (1x200)")
axes[1].set_title("2D trained W1p (tanh)")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel("refinement iterations")
axes[1].legend()

fig.suptitle("2D smooth-disk reproduction (paper-style)")
plt.tight_layout()
plt.show()

## Figure D — 2D local gap heatmaps (paper-style visualization)

In [ ]:
def local_gap_heatmap(model: nn.Module, mode: str, p: float, grid_n: int = 32):
    xs = np.linspace(-1.0, 1.0, grid_n + 1)
    ys = np.linspace(-1.0, 1.0, grid_n + 1)
    out = np.zeros((grid_n, grid_n), dtype=float)
    for i in range(grid_n):
        for j in range(grid_n):
            lo = [float(xs[i]), float(ys[j])]
            hi = [float(xs[i + 1]), float(ys[j + 1])]
            box = IntervalTensor.from_bounds(lo, hi)
            b = model.lpnorm(box, p=p, iterations=0) if mode == "lp" else model.sobolev_norm(box, p=p, iterations=0)
            out[j, i] = float(b.upper - b.lower)
    return out

h_lp_deep = local_gap_heatmap(lp_deep_2d, mode="lp", p=P_VAL, grid_n=24)
h_lp_wide = local_gap_heatmap(lp_wide_2d, mode="lp", p=P_VAL, grid_n=24)
h_w1_deep = local_gap_heatmap(w1p_deep_2d, mode="w1p", p=P_VAL, grid_n=24)
h_w1_wide = local_gap_heatmap(w1p_wide_2d, mode="w1p", p=P_VAL, grid_n=24)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, h, title in [
    (axes[0,0], h_lp_deep, "Lp local gap (deep)"),
    (axes[0,1], h_lp_wide, "Lp local gap (wide)"),
    (axes[1,0], h_w1_deep, "W1p local gap (deep)"),
    (axes[1,1], h_w1_wide, "W1p local gap (wide)"),
]:
    im = ax.imshow(h, origin="lower", extent=[-1,1,-1,1], cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("2D local bound-gap heatmaps")
plt.tight_layout()
plt.show()

## Notes on paper alignment

- Architectures mirror the paper: **deep = 3 hidden layers × 32**, **wide = 1 hidden layer × 200**.
- 1D runs include both **random/untrained** and **trained** settings.
- 2D runs include **trained** settings for both $L^p$ and $W^{1,p}$.
- Figures are designed to be visually similar in spirit: geometric gap decay curves and local-gap heatmaps.
- This notebook intentionally excludes $W^{2,p}$.
